In [0]:
from google.cloud import storage
import pyarrow.parquet as pq


PROJECT_ID = os.environ.get("GCP_PROJECT_ID")
BUCKET_NAME = os.environ.get("GCP_GCS_BUCKET")

In [0]:
def write_read(bucket_name, blob_name):
    """Write and read a blob from GCS using file-like IO"""
    # The ID of your GCS bucket
    # bucket_name = "your-bucket-name"

    # The ID of your new GCS object
    # blob_name = "storage-object-name"

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    # Mode can be specified as wb/rb for bytes mode.
    # See: https://docs.python.org/3/library/io.html
    with blob.open("w") as f:
        f.write("Hello world")

    with blob.open("r") as f:
        print(f.read())


In [0]:
%sql
SELECt current_timestamp

In [0]:

# write data to delta table - spotify_dev.00_bronze
from pyspark.sql.functions import col 

(
    df
    .withColumn("file_source", col("_metadata.file_path"))
    .withColumn("ingestion_time", date_format(current_timestamp(), "dd-MM-yy HH:mm:ss"))
    .writeStream
    .option("checkpointLocation", "/Volumes/spotify_dev/00_landing/spotify_operational_data/_autoloader_checkpoints")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True) # process this structured streaming in batch mode -> process all the files that are present
    .toTable("spotify_dev.01_bronze.playback_test")
)

In [0]:
%sql
SELECT * 
FROM spotify_dev.`01_bronze`.playback_test

In [0]:
%sql
select  date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yy HH:mm:ss");